# JS09: Tugas Lab 2

Pencarian track terdekat pada dataset Micro Spotify menggunakan exact NN, Annoy, FAISS, dan HNSW.

## Instalasi library

In [1]:
%pip install -q annoy==1.17.3 faiss-cpu==1.13.2 hnswlib==0.8.0 requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\syahr\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


## Import library

In [2]:
from pathlib import Path
import requests
import time
import pandas as pd
import numpy as np
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

## Mengunduh dataset pertama

File yang digunakan adalah songs_with_attributes_and_lyrics.csv versi 20, yaitu dataset pertama dari dua file Kaggle pada soal. File lengkap berukuran sekitar 1,55 GB dan disimpan di folder data. Jika belum tersedia, unduhan berjalan otomatis. Unduhan yang terputus dapat dilanjutkan dengan menjalankan ulang sel. Tidak perlu Google Drive atau unggahan manual.

In [3]:
data_path = Path('data/songs_with_attributes_and_lyrics.csv')
data_path.parent.mkdir(parents=True, exist_ok=True)
expected_bytes = 1_545_144_675
dataset_url = ('https://www.kaggle.com/api/v1/datasets/download/bwandowando/'
               'spotify-songs-with-attributes-and-lyrics/'
               'songs_with_attributes_and_lyrics.csv?datasetVersionNumber=20')

if not data_path.exists():
    temporary_path = data_path.with_suffix('.csv.part')
    offset = temporary_path.stat().st_size if temporary_path.exists() else 0
    headers = {'User-Agent': 'Mozilla/5.0', 'Accept-Encoding': 'identity'}
    if offset:
        headers['Range'] = f'bytes={offset}-'
    print('Mengunduh dataset Spotify. Tunggu sampai selesai.', flush=True)
    with requests.get(dataset_url, headers=headers, stream=True, timeout=(60, 180)) as response:
        response.raise_for_status()
        resume = offset > 0 and response.status_code == 206
        with temporary_path.open('ab' if resume else 'wb') as target:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                target.write(chunk)
    if temporary_path.stat().st_size != expected_bytes:
        raise ValueError('Ukuran unduhan tidak sesuai. Jalankan ulang sel unduhan.')
    temporary_path.replace(data_path)

print('Dataset siap:', data_path.name)
print('Ukuran file (byte):', data_path.stat().st_size)

Dataset siap: songs_with_attributes_and_lyrics.csv
Ukuran file (byte): 1545144675


## Memuat CSV dan memilih fitur numerik

Sembilan fitur mengikuti boilerplate: danceability, energy, loudness, speechiness, acousticness, instrumentalness, liveness, valence, dan tempo. Identitas track hanya dipakai untuk menampilkan hasil. Kolom lirik tidak dimuat agar penggunaan memori lebih kecil. Semua baris dibaca, lalu baris dengan fitur kosong atau tidak valid dikeluarkan.

In [4]:
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
identity_columns = ['id', 'name', 'artists']
df = pd.read_csv(data_path, usecols=identity_columns + features, low_memory=False)
initial_rows = len(df)
df[features] = df[features].apply(pd.to_numeric, errors='coerce')
df[features] = df[features].replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=features).reset_index(drop=True)
X = df[features].values
print('Jumlah baris CSV:', initial_rows)
print('Baris fitur tidak valid:', initial_rows - len(df))
print('Ukuran fitur:', X.shape)
print(df[identity_columns + features].head().to_string(index=False))

Jumlah baris CSV: 955320
Baris fitur tidak valid: 0
Ukuran fitur: (955320, 9)
                    id            name  artists  danceability  energy  loudness  speechiness  acousticness  instrumentalness  liveness  valence   tempo
0Prct5TDjAnEgIqbxcldY9               ! HELLYEAH         0.415  0.6050   -11.157       0.0575       0.00116          0.838000    0.4710    0.193 100.059
2ASl4wirkeYm3OWZxXKYuq              !!  Yxngxr1         0.788  0.6480    -9.135       0.3150       0.90000          0.000000    0.1760    0.287  79.998
69lcggVPmOr9cvPx9kLiiN !!! - Interlude   Glowie         0.000  0.0354   -20.151       0.0000       0.90800          0.000000    0.4790    0.000   0.000
4U7dlZjg1s9pjdppqZy0fm  !!De Repente!!  Rosendo         0.657  0.8820    -6.340       0.0385       0.00740          0.000013    0.0474    0.939 123.588
4v1IBp3Y3rpkWmWzIlkYju  !!De Repente!!  Rosendo         0.659  0.8930    -8.531       0.0411       0.09220          0.000019    0.0534    0.951 123.600


## StandardScaler

In [5]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = np.ascontiguousarray(X_scaled, dtype=np.float32)
del X
k = 10
print('Rata rata fitur setelah standardisasi:')
print(pd.Series(X_scaled.mean(axis=0, dtype=np.float64), index=features).round(5).to_string())
print('Simpangan baku fitur setelah standardisasi:')
print(pd.Series(X_scaled.std(axis=0, dtype=np.float64), index=features).round(5).to_string())

Rata rata fitur setelah standardisasi:
danceability        0.0
energy             -0.0
loudness            0.0
speechiness        -0.0
acousticness       -0.0
instrumentalness    0.0
liveness           -0.0
valence             0.0
tempo               0.0
Simpangan baku fitur setelah standardisasi:
danceability        1.0
energy              1.0
loudness            1.0
speechiness         1.0
acousticness        1.0
instrumentalness    1.0
liveness            1.0
valence             1.0
tempo               1.0


## Menentukan query

Track pertama menjadi query. Baris query dikeluarkan dari hasil agar yang ditampilkan adalah sepuluh track lain. Semua metode memakai data standar dan jarak Euclidean yang sama.

In [6]:
query_index = 0
query = X_scaled[query_index:query_index + 1]
search_k = k + 1
print(df.loc[[query_index], identity_columns].to_string(index=False))
comparison = []
neighbor_ids = {}

def select_neighbors(indices):
    result = [int(i) for i in indices if int(i) >= 0 and int(i) != query_index][:k]
    if len(result) != k:
        raise ValueError('Jumlah tetangga belum mencapai sepuluh.')
    return np.asarray(result, dtype=np.int64)

                    id name  artists
0Prct5TDjAnEgIqbxcldY9    ! HELLYEAH


## Exact NN

In [7]:
start = time.perf_counter()
exact_nn = NearestNeighbors(n_neighbors=search_k, algorithm='brute', metric='euclidean')
exact_nn.fit(X_scaled)
build_time = time.perf_counter() - start

start = time.perf_counter()
exact_distances, exact_indices = exact_nn.kneighbors(query)
query_time = time.perf_counter() - start
neighbor_ids['Exact NN'] = select_neighbors(exact_indices[0])
comparison.append({'Metode': 'Exact NN', 'Build (detik)': build_time,
                   'Query (ms)': query_time * 1000})
print('Exact NN selesai.')

Exact NN selesai.


## Annoy

Indeks memakai jarak Euclidean dan 10 pohon.

In [8]:
start = time.perf_counter()
annoy_index = AnnoyIndex(X_scaled.shape[1], 'euclidean')
annoy_index.set_seed(42)
for i, vector in enumerate(X_scaled):
    annoy_index.add_item(i, vector)
annoy_index.build(10)
build_time = time.perf_counter() - start

start = time.perf_counter()
annoy_indices = annoy_index.get_nns_by_vector(query[0], search_k)
query_time = time.perf_counter() - start
neighbor_ids['ANNOY'] = select_neighbors(annoy_indices)
comparison.append({'Metode': 'ANNOY', 'Build (detik)': build_time,
                   'Query (ms)': query_time * 1000})
print('Annoy selesai.')

Annoy selesai.


## FAISS

IVF memakai 100 cluster dan memeriksa 10 cluster terdekat saat query, seperti konfigurasi Lab 5.

In [9]:
start = time.perf_counter()
dim = X_scaled.shape[1]
quantizer = faiss.IndexFlatL2(dim)
faiss_index = faiss.IndexIVFFlat(quantizer, dim, 100, faiss.METRIC_L2)
faiss_index.train(X_scaled)
faiss_index.add(X_scaled)
faiss_index.nprobe = 10
build_time = time.perf_counter() - start

start = time.perf_counter()
faiss_distances, faiss_indices = faiss_index.search(query, search_k)
query_time = time.perf_counter() - start
neighbor_ids['FAISS'] = select_neighbors(faiss_indices[0])
comparison.append({'Metode': 'FAISS', 'Build (detik)': build_time,
                   'Query (ms)': query_time * 1000})
print('FAISS selesai.')

FAISS selesai.


## HNSW

Indeks memakai M=16, ef_construction=200, dan ef=50.

In [10]:
start = time.perf_counter()
hnsw_index = hnswlib.Index(space='l2', dim=X_scaled.shape[1])
hnsw_index.init_index(max_elements=len(X_scaled), ef_construction=200, M=16, random_seed=42)
hnsw_index.add_items(X_scaled)
hnsw_index.set_ef(50)
build_time = time.perf_counter() - start

start = time.perf_counter()
hnsw_indices, hnsw_distances = hnsw_index.knn_query(query, k=search_k)
query_time = time.perf_counter() - start
neighbor_ids['HNSW'] = select_neighbors(hnsw_indices[0])
comparison.append({'Metode': 'HNSW', 'Build (detik)': build_time,
                   'Query (ms)': query_time * 1000})
print('HNSW selesai.')

HNSW selesai.


## Perbandingan performa

In [11]:
ground_truth = set(neighbor_ids['Exact NN'])
for row in comparison:
    row['Recall@10'] = len(set(neighbor_ids[row['Metode']]) & ground_truth) / k
comparison_table = pd.DataFrame(comparison)
print(comparison_table.to_string(index=False, float_format=lambda value: f'{value:.6f}'))

  Metode  Build (detik)  Query (ms)  Recall@10
Exact NN       0.010450  141.312000   1.000000
   ANNOY      14.529568    0.262700   0.700000
   FAISS       0.305578    2.983000   1.000000
    HNSW      67.640981    0.386800   1.000000


## Track terdekat

Jarak pada seluruh tabel dihitung ulang sebagai jarak Euclidean pada fitur standar agar bisa dibandingkan langsung. Recall@10 membandingkan identitas baris dengan hasil exact.

In [12]:
for method, indices in neighbor_ids.items():
    result = df.iloc[indices][identity_columns].copy()
    result.insert(0, 'Peringkat', np.arange(1, k + 1))
    result.insert(1, 'Indeks', indices)
    result['Jarak Euclidean'] = np.linalg.norm(X_scaled[indices] - query[0], axis=1)
    print(f'\n{method}')
    print(result.to_string(index=False, float_format=lambda value: f'{value:.6f}'))


Exact NN
 Peringkat  Indeks                     id                                    name              artists  Jarak Euclidean
         1  394553 2vy246tF9zAM26wZVIr8bx           Intro (Who The Fuck Are You')            C-Lekktor         0.886657
         2  764272 4ikbMYjg2Yn2zjB4TeiX8P                             Testimonium            Lacrimosa         0.909443
         3  837727 1E1M2eJ2o9bFRck0ZHl7Iv Ultimate Satisfaction - The Field Remix       Bear In Heaven         0.936008
         4  749223 2KvIO4ew1TGnz1VfTJk49g                                Sæglópur            Sigur Rós         0.937600
         5   61511 15jU0mzRZHQAI6lhO8cnCw                        Autumn Interlude          Amethystium         0.946505
         6  625227 3jtaAUbFSzfoKv6bVVo563                                 Púrpura  Margaritas Podridas         0.976332
         7  833164 0pcLVqykiOhWTSOcmYJKIb                        Turn from Heaven              Therion         0.995418
         8  676733 1S8jdYiBiX2

## Analisis hasil

In [13]:
fastest = comparison_table.loc[comparison_table['Query (ms)'].idxmin(), 'Metode']
print(f'Query tercepat pada percobaan ini: {fastest}.')
for method in ['ANNOY', 'FAISS', 'HNSW']:
    matches = len(set(neighbor_ids[method]) & ground_truth)
    print(f'{method} menemukan {matches} dari {k} tetangga yang sama dengan exact NN.')

Query tercepat pada percobaan ini: ANNOY.
ANNOY menemukan 7 dari 10 tetangga yang sama dengan exact NN.
FAISS menemukan 10 dari 10 tetangga yang sama dengan exact NN.
HNSW menemukan 10 dari 10 tetangga yang sama dengan exact NN.


StandardScaler menyetarakan skala sembilan fitur sebelum pencarian. Exact NN menjadi pembanding, sedangkan metode ANN dapat mengurangi waktu query dengan biaya pembangunan indeks. Perbandingan ini memakai satu track query. Track dengan fitur identik dapat memiliki jarak sama, sehingga urutan dan identitas pada batas sepuluh tetangga bisa berbeda.